In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import Image
from google.colab import userdata
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.runnables import Runnable
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pathlib import Path
from pydantic import BaseModel, Field, SecretStr
from typing import Annotated, List, TypedDict

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

In [ ]:
class GenerationState(TypedDict):
    topic: str
    code: str
    generation_messages: Annotated[List[BaseMessage], add_messages]
    quality: float | None
    feedback: str | None

In [ ]:
GENERATOR_PROMPT = """You are a helpful coding assistant.
Your task is to generate simple Python code.
"""

EVALUATOR_PROMPT = """You are a pedantic code verifier.
You task is to analyze Python code and point out problems, potential issues or quality improvements.
"""

class Evaluation(BaseModel):
    score: float = Field(ge=0, le=10, description="Overall quality from 0 to 10.")
    feedback: str = Field(description="Actionable feedback.")

generator_model = ChatOpenAI(model="gpt-5-mini", api_key=openai_api_key, reasoning_effort="low")
evaluator_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low").with_structured_output(Evaluation)

In [ ]:
def generate(state: GenerationState):
    feedback = state.get('feedback')
    if feedback:
        task = HumanMessage(feedback)
    else:
        task = HumanMessage(state.get('topic'))

    print("Initiating \"generate\"...")
    response = generator_model.invoke([SystemMessage(GENERATOR_PROMPT), *state.get('generation_messages'), task])

    print("Generated:")
    print(response.content)
    print()

    return { 'code': response.content, 'generation_messages': [task, response], 'quality': None, 'feedback': None }

def evaluate(state: GenerationState):
    print("Initiating \"evaluate\"...")

    response = evaluator_model.invoke([SystemMessage(EVALUATOR_PROMPT), HumanMessage(state.get('code'))])

    print("Evaluation result:")
    print(response)
    print()

    return { 'quality': response.score, 'feedback': response.feedback }

In [ ]:
graph_builder = StateGraph(GenerationState)

graph_builder.add_node("generate", generate)
graph_builder.add_node("evaluate", evaluate)

graph_builder.add_edge(START, "generate")
graph_builder.add_edge("generate", "evaluate")
graph_builder.add_conditional_edges("evaluate", lambda x: "generate" if x.get('quality', 0) < 8 else END, ["generate", END])

graph = graph_builder.compile()

In [ ]:
display_graph(graph, Path("/content/graph.png"))

In [ ]:
final_state = graph.invoke(input={ "topic": "Write an algorithm to find the shortest path in a directed acyclic graph with weighted edges, even when some weights are negative." })

In [ ]:
final_state